# Load Net-Load Data in 4 different Resolutions
The data stems from https://www.nature.com/articles/s41597-022-01156-1.pdf 

We construct realistic net-load timeseries of 15 residential buildings over 2 years and 8 months in 1/15/30/60-min resolution. The buildings only contain load, thuse we scale PV measurements from the same dataset according to each building size (area in m^2) and map PV production on top of each building. We use PV measurements from 3 different locations to increase variability of our data.

In [1]:
import requests
import h5py
import pandas as pd
from io import BytesIO
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import os
import io
import zipfile
#from tqdm import tqdm
from urllib.parse import urlparse

In [ ]:
# hdf5 10sec data
hdf5_2020_10s = 'https://zenodo.org/records/5642902/files/2020_data_10s.zip?download=1'
hdf5_10s = [hdf5_2020_10s] 

#for when we want to download from URL and extract the hdf5 file from the zip
#hdf5_2020_10s_local = 'raw_data/10s/2020_data_10s.zip'


# hdf5 1min data
hdf5_2018 = 'https://zenodo.org/records/5642902/files/2018_data_1min.zip?download=1'
hdf5_2019 = 'https://zenodo.org/records/5642902/files/2019_data_1min.zip?download=1'
hdf5_2020 = 'https://zenodo.org/records/5642902/files/2020_data_1min.zip?download=1'

hdf5_1min = [hdf5_2018, hdf5_2019, hdf5_2020]

# hdf5 15 min data
hdf5_2018_15min = 'https://zenodo.org/records/5642902/files/2018_data_15min.hdf5?download=1'
hdf5_2019_15min = 'https://zenodo.org/records/5642902/files/2019_data_15min.hdf5?download=1'
hdf5_2020_15min = 'https://zenodo.org/records/5642902/files/2020_data_15min.hdf5?download=1'

hdf5_15min = [hdf5_2018_15min, hdf5_2019_15min, hdf5_2020_15min]

# we obtain 30min data by resampling 15min data

# hdf5 1h data
hdf5_2018_1h = 'https://zenodo.org/records/5642902/files/2018_data_60min.hdf5?download=1'
hdf5_2019_1h = 'https://zenodo.org/records/5642902/files/2019_data_60min.hdf5?download=1'
hdf5_2020_1h = 'https://zenodo.org/records/5642902/files/2020_data_60min.hdf5?download=1'

hdf5_1h = [hdf5_2018_1h, hdf5_2019_1h, hdf5_2020_1h]

In [ ]:
def download_hdf5(url: str) -> BytesIO:
    """
    Downloads an HDF5 file from a URL with a progress bar and returns a BytesIO buffer.
    """
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get('content-length', 0))
    chunk_size = 1024 * 1024  # 1 MB
    buffer = BytesIO()

    with tqdm(total=total_size, unit='B', unit_scale=True, unit_divisor=1024,
              desc=f"Downloading {url.split('/')[-1]}") as pbar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:  # filter out keep-alive chunks
                buffer.write(chunk)
                pbar.update(len(chunk))

    buffer.seek(0)  # Reset pointer to beginning
    return buffer

In [ ]:
import os
import zipfile
import io
from urllib.parse import urlparse
from tqdm import tqdm

def download_all_data(resolution_name, urls, save_dir):
    """
    Downloads all files for a given resolution. Detects and extracts HDF5 files from ZIPs.
    Returns a list of local file paths to prevent high RAM usage.
    """
    print(f"\nDownloading {resolution_name} data...")
    os.makedirs(save_dir, exist_ok=True)
    
    hdf5_filepaths = []

    for hdf_url in tqdm(urls, desc=f"{resolution_name}"):
        local_filename = os.path.join(save_dir, os.path.basename(urlparse(hdf_url).path))

        # Download file if it doesn't exist locally
        if os.path.exists(local_filename):
            tqdm.write(f"Skipping (already exists): {local_filename}")
        else:
            raw_buffer = download_hdf5(hdf_url)
            with open(local_filename, 'wb') as f:
                f.write(raw_buffer.getbuffer())

        # Check for ZIP signature (first 4 bytes)
        with open(local_filename, 'rb') as f:
            file_header = f.read(4)

        if file_header == b'PK\x03\x04': 
            with zipfile.ZipFile(local_filename, 'r') as zf:
                for file_info in zf.infolist():
                    if file_info.filename.endswith(('.h5', '.hdf5')):
                        extracted_path = os.path.join(save_dir, file_info.filename)
                        
                        # Extract to disk if not already present
                        if not os.path.exists(extracted_path):
                            tqdm.write(f"Extracting {file_info.filename} to disk...")
                            zf.extract(file_info, save_dir)
                            
                        hdf5_filepaths.append(extracted_path)
        else:
            hdf5_filepaths.append(local_filename)

    print(f"{len(hdf5_filepaths)} HDF5 files ready for use on disk.")
    return hdf5_filepaths


data_10s = download_all_data("10 second", hdf5_10s, "raw_data/10s")
data_1min = download_all_data("1 minute", hdf5_1min, "raw_data/1min")
data_15min = download_all_data("15 minute", hdf5_15min, "raw_data/15min")
data_1h = download_all_data("1 hour", hdf5_1h, "raw_data/60min")

10 second:   0%|          | 0/1 [09:26<?, ?it/s]


KeyboardInterrupt: 

## Helper Functions


In [1]:
def concat_dataframes_unique_index(df_list):
    # Combine all indices into a single Series
    all_indices = pd.concat([df.index.to_series() for df in df_list], ignore_index=True)

    # Convert to UTC datetime
    utc_indices = pd.to_datetime(all_indices, unit='s', utc=True)


    # Sanity check for UTC conversion
    assert utc_indices.notnull().all(), "Some indices could not be converted to UTC timestamps"
    assert str(utc_indices.dt.tz) == 'UTC', "Indices are not in UTC timezone."
    # Sanity check for duplicates
    assert len(utc_indices) == utc_indices.nunique(), "Duplicate index values found across dataframes."
    # Check if all indices are sorted
    assert utc_indices.is_monotonic_increasing, "Indices are not sorted in increasing order."
    # Check if they are equally spaced
    time_diffs = utc_indices.diff().dropna()
    assert time_diffs.nunique() == 1, "Indices are not equally spaced."

    # Concatenate dataframes
    df_full = pd.concat(df_list, axis=0, ignore_index=True)

    # Replace the index with the UTC-converted version
    df_full.index = utc_indices

    return df_full


def get_data_from_buffer(file_group, data_list):
    """
    Reads HDF5 data from a BytesIO buffer and appends it to a list of DataFrames.
    """
    dfs = []
    for hdf_buf in data_list:
        with h5py.File(hdf_buf, "r") as h5file:
            # Navigate to the table
            table = h5file[file_group]

            # Or convert to structured array or DataFrame
            df = pd.DataFrame.from_records(table[:])
            df.set_index('index', inplace=True)
            dfs.append(df)

    df_full = concat_dataframes_unique_index(dfs)
    return df_full


def category_data(categorys, data, is_pv = False):

    categorys_data = {}
    for category in categorys:
        print(f"Loading {category}...")
        if is_pv:
            name = category.split('/')[4]
        else:
            name = category.split('/')[1]

        try:
            df = get_data_from_buffer(category, data)
            categorys_data[name] = df
        except Exception as e:
            print(f"Error loading {category}: {e}")

    return categorys_data


def get_metadata(meta_items, data):
    """
    Reads metadata from a list of items in the HDF5 file and returns a dictionary.
    """
    metadata = {}
    for item in meta_items:
        print(f"Loading metadata for {item}...")

        item_name = item.split('/')[1]  # Extract the item name from the path
        try:
            with h5py.File(data[0], "r") as hdf:
                meta = hdf[item]
                metadata[item_name] = dict(meta.attrs)
        except Exception as e:
            print(f"Error loading metadata for {item}: {e}")
    return metadata

## Extract relevant data and map PV onto Buildings

In [ ]:
hdf5_buffer = data_1min[0]  # Use the first buffer for interfering building list

list_of_items = []
with h5py.File(hdf5_buffer, "r") as hdf:
    def print_structure(name, obj):
        print(name, dict(obj.attrs))
        list_of_items.append(name)
    hdf.visititems(print_structure)

In [ ]:
# filter the list of items for everthing contain table
table_items = [item for item in list_of_items if 'table' in item]
no_table_items = [item for item in list_of_items if 'table' not in item]
# Categorize the items based on their type
heatpumps = [item for item in table_items if 'HEATPUMP' in item]
buildings = [item for item in table_items if 'HOUSEHOLD' in item]
misc = [item for item in table_items if 'PV1' in item]

meta_houshold = [item for item in no_table_items if 'HOUSEHOLD' in item]


#
# !!!!! Panel Number taken from Dataset paper linked above !!!!! 
#
meta_solar_data = {
    "SOUTH": 58, # Number of PV Panels resulting in the respective measure. We assume that each panel contributes equally to the total measurement.
    "EAST": 78,
    "WEST": 78,
}

In [ ]:
# 10 sec data
building_data_10s = category_data(buildings, data_10s)
heatpump_data_10s = category_data(heatpumps, data_10s)
misc_data_10s = category_data(misc, data_10s, is_pv=True)
meta_houshold_data_10s = get_metadata(meta_houshold, data_10s)

# 1 min data
building_data_1min = category_data(buildings, data_1min)
heatpump_data_1min = category_data(heatpumps, data_1min)
misc_data_1min = category_data(misc, data_1min, is_pv=True)
meta_houshold_data_1min = get_metadata(meta_houshold, data_1min)

# 15 min data
building_data_15min = category_data(buildings, data_15min)
heatpump_data_15min = category_data(heatpumps, data_15min)
misc_data_15min = category_data(misc, data_15min, is_pv=True)
meta_houshold_data_15min = get_metadata(meta_houshold, data_15min)

# 1 hour data
building_data_1h = category_data(buildings, data_1h)
heatpump_data_1h = category_data(heatpumps, data_1h)
misc_data_1h = category_data(misc, data_1h, is_pv=True)
meta_houshold_data_1h = get_metadata(meta_houshold, data_1h)

#############################################################
# generate 30 min data via resampling

building_data_30min = {}
heatpump_data_30min = {}
misc_data_30min = {}
meta_houshold_data_30min = {}


for key in building_data_15min.keys():
    building_data_30min[key] = building_data_15min[key].resample('30min').mean()
    heatpump_data_30min[key] = heatpump_data_15min[key].resample('30min').mean()

for key in misc_data_15min.keys():
    misc_data_30min[key] = misc_data_15min[key].resample('30min').mean()

for key in meta_houshold_data_15min.keys():
    meta_houshold_data_30min[key] = meta_houshold_data_15min[key]

In [ ]:
def trim_dataframe_from_valid_window(df: pd.DataFrame, window_size: int) -> pd.DataFrame:
    if not isinstance(df, pd.DataFrame):
        raise ValueError("Input must be a pandas DataFrame.")
    if window_size < 1:
        raise ValueError("Window size must be at least 1.")
    
    row_valid_mask = df.notna().all(axis=1).astype(int)
    valid_windows = row_valid_mask.rolling(window=window_size, min_periods=window_size).sum()
    first_valid_start = valid_windows[valid_windows == window_size].index.min()

    if pd.notna(first_valid_start):
        return df.loc[first_valid_start:]
    else:
        return pd.DataFrame(columns=df.columns, index=df.index[:0])

def generate_prosumption_df(buidding_data, heatpump_data, misc_data, building_name, misc_key='EAST', pv_factor=0.5, heatpump_factor=1.0):
    prosumption = (buidding_data[building_name]['P_TOT'] +
               heatpump_factor * heatpump_data[building_name]['P_TOT'] -
               pv_factor * misc_data[misc_key]['P_TOT'])
    
    interferred_resolution = prosumption.index.inferred_freq if prosumption.index.inferred_freq else 'unknown'
    
    # --- FIXED FOR 10s ---
    if interferred_resolution == 'min':
        window_size = 24 * 7 * 60 
    elif interferred_resolution in ['10s', '10S']:  # <--- Added 10-second support
        window_size = 24 * 7 * 60 * 6  # 6 intervals per minute * 60 * 24 * 7
    elif interferred_resolution == '15min':
        window_size = 24 * 7 * 4 
    elif interferred_resolution == '30min':
        window_size = 24 * 7 * 2 
    elif interferred_resolution == 'h':
        window_size = 24 * 7 
    else:
        raise ValueError(f"Unsupported resolution: {interferred_resolution}")

    prosumption_df= trim_dataframe_from_valid_window(pd.DataFrame(prosumption), window_size=window_size)  

    prosumption_df["Building P_TOT"] = buidding_data[building_name]['P_TOT']
    prosumption_df["Heatpump P_TOT"] = heatpump_factor * heatpump_data[building_name]['P_TOT']
    prosumption_df["Solar P_TOT"] = pv_factor * misc_data[misc_key]['P_TOT']

    return prosumption_df

In [ ]:
def vizualize_prosumption(prosumption_df, title='Prosumption Data', figsize=(10, 5)):
    """
    Visualizes the prosumption data.
    """
    #prosumption_df["P_TOT"].plot(title='Prosumption', figsize=(10, 5))
    # plt.xlabel('Time')
    # plt.ylabel('Power (W)')
    # plt.title(title)
    # plt.grid()
    # plt.show()

    #
    # Train Val Test split visualization as static Example
    #

    # Splits the prosumption data
    # first year train, second year validation, third year test

    start = prosumption_df.index[0]
    train_end = start + pd.DateOffset(years=1)
    val_end = train_end + pd.DateOffset(years=1)
    end = prosumption_df.index[-1]

    display(f"Train period: {start} to {train_end}  contains {train_end - start}")
    display(f"Validation period: {train_end} to {val_end}  contains {val_end - train_end}") 
    display(f"Test period: {val_end} to {end} contains {prosumption_df.index[-1] - val_end}")

    # vizualize the splits as marked zones in the plot
    prosumption_df["P_TOT"].plot(title='Prosumption with splits', figsize=(10, 5))
    plt.axvspan(start, train_end, color='green', alpha=0.3, label='Train')
    plt.axvspan(train_end, val_end, color='orange', alpha=0.3, label='Validation')
    plt.axvspan(val_end, prosumption_df.index[-1], color='red', alpha=0.3, label='Test')
    plt.legend()
    plt.show()  


    #
    # May 2019 visualization as static example
    #

    prosumption_df.loc['2019-05-01':'2019-06-01'].plot(title='Prosumption in May 2019', figsize=(10, 5))
    plt.xlabel('Time')
    plt.ylabel('Power (W)')
    plt.title(title + ' in May 2019')
    plt.grid()
    plt.show()

## Generate houshold datasets

In [ ]:
def save_prosumption_df(prosumption_df, building_name, misc_key , num_pv_modules, heatpump_factor=1.0):
    if not pd.api.types.is_datetime64_any_dtype(prosumption_df.index):
        raise ValueError("The DataFrame index must be a datetime index.")

    directory = 'prosumption_data'
    if not os.path.exists(directory):
        os.makedirs(directory)  

    resolution = prosumption_df.index.inferred_freq if prosumption_df.index.inferred_freq else 'unknown'
    
    # --- FIXED FOR 10s ---
    if resolution == 'h':
        resolution = '60min'
    elif resolution in ['10s', '10S']: # <--- Added 10-second folder naming
        resolution = '10sec'
    elif resolution == '15min':
        resolution = '15min'
    elif resolution == '30min':
        resolution = '30min'
    elif resolution == 'min':
        resolution = '1min'
    else:
        raise ValueError(f"Unsupported resolution: {resolution}")

    resolution_dir = os.path.join(directory, resolution)
    if not os.path.exists(resolution_dir):
        os.makedirs(resolution_dir)
    
    filename = f"prosumption_data/{resolution}/prosumption_{building_name}_num_pv_modules_{num_pv_modules}_pv_{misc_key}_hp_{heatpump_factor}.csv"
    prosumption_df.to_csv(filename)
    print(f"Prosumption data for {building_name} saved to {filename}")
    return filename

def calculate_and_save_prosumption(building_data , heatpump_data, misc_data, meta_data, meta_solar_data, building_name, misc_key='EAST', pv_factor=None, heatpump_factor=None , verbose=False):
    if verbose: 
        print("----------------------------------------------------")
        print(f"Calculating prosumption DataFrame for {building_name} with misc_key {misc_key}...")

    if pv_factor is None:
        pv_factor, num_modules = get_pv_factor(building_name, meta_data, meta_solar_data, misc_key)
    if heatpump_factor is None:
        heatpump_factor = 1.0

    prosumption_df = generate_prosumption_df(building_data, heatpump_data, misc_data, building_name, misc_key, pv_factor, heatpump_factor)
    filename = save_prosumption_df(prosumption_df, building_name, misc_key, num_modules, heatpump_factor)

    if verbose:
        print(f"Prosumption DataFrame for {building_name} saved to {filename}")
        display(prosumption_df.describe())
        vizualize_prosumption(prosumption_df, title=f'Prosumption {building_name} Data')
        
    return filename

def get_pv_factor(building_name, meta_data, meta_solar_data, misc_key='EAST'):
    living_space = meta_data[building_name]['living_space']
    pv_size_modules = 1.6
    usable_rooftop_space = 0.3 
    modules = meta_solar_data[misc_key]
    modules_rooftop = int((usable_rooftop_space * living_space) / pv_size_modules)
    pv_factor = modules_rooftop / modules
    print(f"Calculated PV factor for {building_name} is {pv_factor:.2f}")
    return pv_factor, modules_rooftop

In [ ]:
# Candidate household numbers
household_numbers = [3, 4, 9, 10, 12, 14, 16, 18, 19, 22, 27, 28, 29, 30, 32, 36]  # Nr 10 (SFH10) only has data available until end of november 2020 and has therefore been excluded from further analysis.
# SFH List
sfh_list = [f"SFH{num}" for num in household_numbers]

In [ ]:
# The first households are south facing, the second ones are east facing, the last ones are west facing
south_facing = sfh_list[0:6]
east_facing = sfh_list[6:10]
west_facing = sfh_list[10:16]

for sfh in south_facing:
    calculate_and_save_prosumption(building_data=building_data_1min, heatpump_data=heatpump_data_1min, misc_data=misc_data_1min, meta_data=meta_houshold_data_1min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='SOUTH',  heatpump_factor=1.0, verbose=True)
for sfh in east_facing:
    calculate_and_save_prosumption(building_data=building_data_1min, heatpump_data=heatpump_data_1min, misc_data=misc_data_1min,  meta_data=meta_houshold_data_1min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='EAST',  heatpump_factor=1.0, verbose=True)
for sfh in west_facing:
    calculate_and_save_prosumption(building_data=building_data_1min, heatpump_data=heatpump_data_1min, misc_data=misc_data_1min,  meta_data=meta_houshold_data_1min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='WEST',  heatpump_factor=1.0, verbose=True)

In [ ]:
# 15 min data
for sfh in south_facing:
    calculate_and_save_prosumption(building_data=building_data_15min, heatpump_data=heatpump_data_15min, misc_data=misc_data_15min, meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='SOUTH',  heatpump_factor=1.0, verbose=True)
for sfh in east_facing:
    calculate_and_save_prosumption(building_data=building_data_15min, heatpump_data=heatpump_data_15min, misc_data=misc_data_15min,  meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='EAST',  heatpump_factor=1.0, verbose=True)
for sfh in west_facing:
    calculate_and_save_prosumption(building_data=building_data_15min, heatpump_data=heatpump_data_15min, misc_data=misc_data_15min,  meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='WEST',  heatpump_factor=1.0, verbose=True)

In [ ]:
# 1h data
for sfh in south_facing:
    calculate_and_save_prosumption(building_data=building_data_1h, heatpump_data=heatpump_data_1h, misc_data=misc_data_1h, meta_data=meta_houshold_data_1h, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='SOUTH',  heatpump_factor=1.0, verbose=True)
for sfh in east_facing:
    calculate_and_save_prosumption(building_data=building_data_1h, heatpump_data=heatpump_data_1h, misc_data=misc_data_1h,  meta_data=meta_houshold_data_1h, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='EAST',  heatpump_factor=1.0, verbose=True)
for sfh in west_facing:
    calculate_and_save_prosumption(building_data=building_data_1h, heatpump_data=heatpump_data_1h, misc_data=misc_data_1h,  meta_data=meta_houshold_data_1h, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='WEST',  heatpump_factor=1.0, verbose=True)

In [ ]:
# 30 min data

# BEWARE THE META DATA FOR 15 min is used

for sfh in south_facing:
    calculate_and_save_prosumption(building_data=building_data_30min, heatpump_data=heatpump_data_30min, misc_data=misc_data_30min, meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='SOUTH',  heatpump_factor=1.0, verbose=True)
for sfh in east_facing:
    calculate_and_save_prosumption(building_data=building_data_30min, heatpump_data=heatpump_data_30min, misc_data=misc_data_30min,  meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='EAST',  heatpump_factor=1.0, verbose=True)
for sfh in west_facing:
    calculate_and_save_prosumption(building_data=building_data_30min, heatpump_data=heatpump_data_30min, misc_data=misc_data_30min,  meta_data=meta_houshold_data_15min, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='WEST',  heatpump_factor=1.0, verbose=True)

In [ ]:
# 10 sec data processing
print("\n=== Processing 10-Second Resolution Data ===")

for sfh in south_facing:
    calculate_and_save_prosumption(building_data=building_data_10s, heatpump_data=heatpump_data_10s, misc_data=misc_data_10s, meta_data=meta_houshold_data_10s, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='SOUTH',  heatpump_factor=1.0, verbose=True)
for sfh in east_facing:
    calculate_and_save_prosumption(building_data=building_data_10s, heatpump_data=heatpump_data_10s, misc_data=misc_data_10s,  meta_data=meta_houshold_data_10s, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='EAST',  heatpump_factor=1.0, verbose=True)
for sfh in west_facing:
    calculate_and_save_prosumption(building_data=building_data_10s, heatpump_data=heatpump_data_10s, misc_data=misc_data_10s,  meta_data=meta_houshold_data_10s, meta_solar_data=meta_solar_data, building_name=sfh , misc_key='WEST',  heatpump_factor=1.0, verbose=True)